In [0]:

source_path = "/Volumes/job_market/landing/raw_files/hacker_news_api/"

checkpoint_path = "/Volumes/job_market/landing/raw_files/_pipeline_state/hn_api_bronze_v1/"

bronze_table = "job_market.bronze.hacker_news_api_raw"

display(dbutils.fs.ls(source_path))

In [0]:

from pyspark.sql import functions as F

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

df = (
    df
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("ingested_at", F.current_timestamp())
)


In [0]:

query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print("Bronze Ingestion abgeschlossen!")


In [0]:
%sql

SELECT
    thread_month,
    COUNT(*) AS total_comments,
    COUNT(DISTINCT item.id) AS unique_posts,
    COUNT_IF(item.deleted = true) AS deleted_posts,
    COUNT_IF(item.dead = true) AS dead_posts,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
FROM job_market.bronze.hacker_news_api_raw
GROUP BY thread_month
ORDER BY thread_month;
